# Évaluation finale 2025 : les deux candidats
**Contributeurs :** Cécile · **Date :** 14/11/2026

Ce notebook présente le résultat du test final sur 2025, l'année réservée au test depuis le début du projet. Il **ne recalcule rien et n'ouvre pas 2025** : il lit les fichiers de résultats écrits par les deux scripts de ce dossier.

> **À lancer avant ce notebook** (une fois, depuis la racine du projet) : les deux scripts, qui produisent les fichiers CSV lus ici.
> ```bash
> python 03_modelisation/4_evaluation_2025/modele_hierarchique_test_final.py
> python 03_modelisation/4_evaluation_2025/modele_keras_test_final.py
> ```
> Chaque script vérifie d'abord qu'il redonne le résultat de 2024 ; sinon il s'arrête sans ouvrir 2025.

**Question :** les deux candidats retenus à la fin du niveau 2 battent-ils la naïve en 2025 ?

## 1. Les déclarations
Chaque candidat a été déclaré **avant l'ouverture de 2025** (12/11/2026), dans l'en-tête de son script. Les résultats sont rapportés tels quels : aucune variante n'a été essayée sur 2025.

| | Candidat agrégé | Candidat par vente |
|---|---|---|
| **Modèle** | Hiérarchique, ordonnée aléatoire par sous-bassin, variables du pont + taux d'effort (H6, modèle 2.2) | H7v4 : réseau Keras optimisé, variables de H7v3 + volume et élan + capacité d'achat + trimestres précédents |
| **Référence (naïve)** | prix N = prix N-1 | prix médian N-1 du sous-bassin |
| **Évaluations** | une seule, sur le prix médian du sous-bassin | **B** : chaque vente ; **C** : médiane du sous-bassin, panier fixe |

**Critère de réussite (le même pour tous, par type de bien) :** gain sur la naïve d'au moins 3 %, IC 95 % haut du bootstrap sous 0 et rang-bisériale négative.

## 2. Chargement
La cellule vérifie que les deux fichiers de résultats existent. S'il en manque un, elle dit quel script lancer.

In [1]:
import pandas as pd

import commun

SCRIPTS = {
    "modele_hierarchique_test_final_resultats.csv": "modele_hierarchique_test_final.py",
    "modele_keras_test_final_resultats.csv": "modele_keras_test_final.py",
}
manquants = [f for f in SCRIPTS if not (commun.DATA_OUT / f).exists()]
assert not manquants, ("Fichiers de résultats absents : " + ", ".join(manquants)
                       + ". À lancer d'abord, depuis la racine du projet :\n"
                       + "\n".join(f"    python 03_modelisation/4_evaluation_2025/{SCRIPTS[f]}" for f in manquants))

agrege = pd.read_csv(commun.DATA_OUT / "modele_hierarchique_test_final_resultats.csv")
par_vente = pd.read_csv(commun.DATA_OUT / "modele_keras_test_final_resultats.csv")

COLONNES = ["MAE (€/m²)", "MAE naïve (€/m²)", "gain sur la naïve (%)", "IC 95 % bas (€/m²)",
            "IC 95 % haut (€/m²)", "rang-bisériale", "bassins où le modèle gagne (%)"]
print("Candidat agrégé :", len(agrege), "lignes | candidat par vente :", len(par_vente), "lignes")

Candidat agrégé : 2 lignes | candidat par vente : 4 lignes


**Lecture.** Le candidat agrégé (modèle hiérarchique) a 2 lignes : une par type de bien. Le candidat par vente (Keras) en a 4 : deux types de bien, chacun évalué en B (biens vendus en 2025) et en C (panier fixe). C'est l'évaluation C qui se compare au candidat agrégé : même information, même mesure.

## 3. Candidat agrégé : modèle hiérarchique
Un seul tableau, une ligne par type de bien.
- **MAE** : erreur moyenne absolue du modèle, en €/m², et celle de la naïve.
- **Gain sur la naïve** : positif, le modèle fait mieux que la naïve ; négatif, moins bien.
- **IC 95 %** (bootstrap par bassin de vie) : la fourchette de l'écart de MAE avec la naïve, en tenant compte du hasard. Entièrement sous 0, le modèle fait vraiment mieux ; si elle contient 0, on ne peut pas conclure ; entièrement au-dessus, il fait vraiment moins bien.
- **Rang-bisériale** : de −1 (le modèle fait mieux partout) à +1 (moins bien partout) ; autour de 0, pas de différence nette.
- **Biais** : prévision moins prix réel, en moyenne (positif : le modèle surestime).
- **Évolution médiane du prix 2025** : prix N / prix N-1 − 1, médiane des sous-bassins : le contexte du marché en 2025.
- **Réussi** : les trois conditions du critère sont remplies.

In [2]:
agrege[["type de bien"] + COLONNES + ["biais (€/m²)", "évolution médiane du prix 2025 (%)", "réussi"]].round(2)

,type de bien,MAE (€/m²),MAE naïve (€/m²),gain sur la naïve (%),IC 95 % bas (€/m²),IC 95 % haut (€/m²),rang-bisériale,bassins où le modèle gagne (%),biais (€/m²),évolution médiane du prix 2025 (%),réussi
0,appartement,147.62,141.77,-4.13,-0.34,12.6,0.03,50.3,-23.53,1.54,False
1,maison,109.20,103.45,-5.56,3.04,9.1,0.10,47.1,-20.58,0.87,False


**Lecture du candidat agrégé.** Le modèle hiérarchique ne bat pas la naïve en 2025.

- **Appartements** : MAE de 147,6 €/m² contre 141,8 pour la naïve, soit −4,1 %. L'IC 95 % [−0,3 ; 12,6] contient 0 : pas de différence démontrée. Le modèle gagne dans 50,3 % des bassins, la rang-bisériale est à 0,03 : match nul, léger désavantage en MAE.
- **Maisons** : MAE de 109,2 €/m² contre 103,5, soit −5,6 %. L'IC 95 % [3,0 ; 9,1] est entièrement au-dessus de 0 : le modèle est significativement moins bon que la naïve. Il gagne dans 47,1 % des bassins, la rang-bisériale est à 0,10.
- **Règle de décision** : année non réussie pour les deux types de bien (IC haut non négatif, rang-bisériale positive, gain sous 3 %).

Le prix médian varie peu en 2025 (+1,5 % pour les appartements, +0,9 % pour les maisons) : la naïve est déjà très bonne dans un marché presque plat. Le biais est négatif (−23,5 et −20,6 €/m²) : le modèle sous-prévoit en moyenne. Hypothèse, non vérifiée ici : il prévoit trop de baisse.

## 4. Candidat par vente : H7v4
Le même tableau, avec deux évaluations :
- **B** : chaque vente de 2025 est prévue, et l'erreur est mesurée sur le prix de la vente ; la naïve est le prix médian N-1 du sous-bassin de la vente. Le biais est un écart moyen pondéré, en %.
- **C** : le réseau prévoit le prix des biens vendus en 2024 (panier fixe) ; la médiane pondérée de ces prévisions est comparée au prix médian réel de 2025 du sous-bassin, comme pour un modèle agrégé. Le biais est l'écart de médiane, en €/m².

In [3]:
(par_vente.sort_values(["évaluation", "type de bien"])
 [["évaluation", "type de bien"] + COLONNES + ["biais", "réussi"]].round(2))

,évaluation,type de bien,MAE (€/m²),MAE naïve (€/m²),gain sur la naïve (%),IC 95 % bas (€/m²),IC 95 % haut (€/m²),rang-bisériale,bassins où le modèle gagne (%),biais,réussi
0,B,appartement,703.30,751.42,6.41,-56.77,-39.68,-0.51,71.83,+0.7 %,True
1,B,maison,672.10,729.80,7.91,-64.75,-51.19,-0.58,72.45,+3.9 %,True
2,C,appartement,185.83,140.66,-32.11,31.10,59.12,0.33,38.40,-14 €/m²,False
3,C,maison,159.02,103.35,-53.86,48.74,62.51,0.45,34.22,-1 €/m²,False


**Lecture du candidat par vente.** Le résultat dépend de l'évaluation.

- **B (biens vendus en 2025)** : le modèle bat la naïve pour les deux types de bien. Appartements : MAE de 703,3 contre 751,4 €/m², gain de 6,4 %, IC 95 % [−56,8 ; −39,7]. Maisons : 672,1 contre 729,8, gain de 7,9 %, IC 95 % [−64,8 ; −51,2]. Les IC sont entièrement sous 0, la rang-bisériale vaut −0,51 et −0,58, et le modèle gagne dans 72 % des bassins. Année réussie pour les deux types.
- **C (panier fixe)** : le modèle perd nettement. Appartements : 185,8 contre 140,7 €/m², soit −32,1 %. Maisons : 159,0 contre 103,4, soit −53,9 %. Les IC sont entièrement au-dessus de 0 ([31,1 ; 59,1] et [48,7 ; 62,5]), la rang-bisériale vaut +0,33 et +0,45, et le modèle gagne dans 34 à 38 % des bassins. Année non réussie pour les deux types.

La naïve n'est pas la même dans les deux évaluations. En B, c'est le prix médian N-1 du sous-bassin comparé à chaque vente : elle ignore les caractéristiques du bien (surface, pièces, terrain), que le modèle par vente connaît. Le gain en B vient donc au moins en partie de cette information. En C, la comparaison porte sur la médiane du sous-bassin, à information égale avec le candidat agrégé : c'est la seule évaluation qui juge la prévision du prix du territoire.

Le biais est exprimé en % pour B et en €/m² pour C : les deux lignes ne se comparent pas.

## 5. Comparaison des deux candidats
Seule l'**évaluation C** du candidat par vente se compare au candidat agrégé : elle juge aussi la médiane du sous-bassin. L'évaluation B juge chaque vente, une autre unité : on ne la compare pas à l'agrégé.

Il faut comparer le **gain de chaque modèle contre sa propre naïve** plus que les MAE entre elles. Les deux naïves ne sont pas exactement les mêmes, et la colonne « bassins de vie » dit combien de bassins chaque évaluation couvre : si les deux nombres diffèrent, les deux modèles ne sont pas jugés sur exactement les mêmes territoires.

In [4]:
c = par_vente[par_vente["évaluation"] == "C"]
comparaison = agrege.merge(c, on="type de bien", suffixes=(" (agrégé)", " (par vente, C)"))
colonnes = ["MAE (€/m²)", "MAE naïve (€/m²)", "gain sur la naïve (%)", "bassins de vie", "réussi"]
(comparaison.set_index("type de bien")
 [[f"{k} ({v})" for k in colonnes for v in ("agrégé", "par vente, C")]].round(2))

,MAE (€/m²) (agrégé),"MAE (€/m²) (par vente, C)",MAE naïve (€/m²) (agrégé),"MAE naïve (€/m²) (par vente, C)",gain sur la naïve (%) (agrégé),"gain sur la naïve (%) (par vente, C)",bassins de vie (agrégé),"bassins de vie (par vente, C)",réussi (agrégé),"réussi (par vente, C)"
type de bien,,,,,,,,,,
appartement,147.62,185.83,141.77,140.66,-4.13,-32.11,695,685,False,False
maison,109.20,159.02,103.45,103.35,-5.56,-53.86,1606,1603,False,False


**Lecture de la comparaison (agrégé contre par vente, évaluation C).** À information égale, le modèle hiérarchique fait mieux que le modèle par vente en 2025, pour les deux types de bien.

| Type de bien | MAE agrégé (€/m²) | MAE par vente, C (€/m²) | Gain agrégé | Gain par vente, C |
|---|---|---|---|---|
| Appartement | 147,6 | 185,8 | −4,1 % | −32,1 % |
| Maison | 109,2 | 159,0 | −5,6 % | −53,9 % |

- Les naïves sont presque identiques (140,7 contre 141,8 €/m² pour les appartements, 103,4 contre 103,5 pour les maisons) : les deux évaluations se font sur des territoires très proches, la comparaison est valable. Les bassins de vie diffèrent un peu (695 contre 685 pour les appartements, 1 606 contre 1 603 pour les maisons) : 10 bassins d'appartements et 3 de maisons ne sont évalués que dans le candidat agrégé.
- Aucun des deux candidats n'est réussi en C. Le modèle par vente perd beaucoup plus que le modèle hiérarchique : 32 à 54 % d'erreur en plus que la naïve, contre 4 à 6 %.
- Le modèle par vente n'est réussi qu'en B, contre une naïve qui ignore les caractéristiques du bien : ce succès ne démontre pas une meilleure prévision du prix du territoire.

## Conclusion

**Un candidat sur deux réussit, et pas sur la même mesure.**

- **Modèle par vente H7v4 (Keras)** : réussi en évaluation B pour les deux types de bien, contre une naïve qui donne à chaque vente le prix médian N-1 de son sous-bassin. Gain de 6,4 % pour les appartements et de 7,9 % pour les maisons, intervalles entièrement sous 0, rang-bisériale de −0,51 et −0,58, gain dans 72 % des sous-bassins. Ces gains sont du même ordre que ceux de 2023 et 2024 (5,4 à 7,3 %). Biais faible (+0,7 % et +3,9 %). H7v4 est le seul modèle du projet qui bat la naïve les trois années jugées ; c'est le modèle retenu pour l'outil Compagnon Immobilier.
- **Pour la médiane d'un sous-bassin, aucun modèle ne bat la naïve en 2025.**
  - Hiérarchique + taux d'effort : −4,1 % pour les appartements (IC 95 % [−0,3 ; 12,6], pas d'écart net) et −5,6 % pour les maisons (IC 95 % [3,0 ; 9,1], significativement moins bon).
  - H7v4 en évaluation C : −32,1 % et −53,9 %. Son biais est presque nul (−14 et −1 €/m²), l'erreur vient donc de l'écart propre à chaque sous-bassin.
  - En C, le modèle hiérarchique fait mieux que H7v4 (147,6 contre 185,8 €/m² pour les appartements, 109,2 contre 159,0 pour les maisons).

**Lecture.** H7v4 prévoit le prix d'une vente, pas l'évolution d'un marché. En B, sa naïve ignore les caractéristiques du bien, que le réseau connaît : une part du gain peut venir de cette information, ce test ne la mesure pas. L'évaluation C, à information égale avec le candidat agrégé, est celle qui juge la prévision du prix du territoire, et H7v4 y échoue.

**Limites.** Un seul millésime de test, dans un marché presque plat (+1,5 % pour les appartements, +0,9 % pour les maisons) où la naïve est très difficile à battre. Les modèles sont réappris sur 2017-2024 et jugés une seule fois sur 2025, avec les déclarations du 12/11, sans modification après coup. De nombreux modèles sont comparés sans correction des comparaisons multiples.